In [86]:
pip install wget

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [87]:
import os
import wget
import shutil 
import base64
import pandas as pd
from selenium import webdriver

### **Initialize the web driver object**

In [88]:
driver = webdriver.Chrome()

In [89]:
driver.get("https://www.amazon.eg/s?k=mobile+phones+egypt&language=en_AE&adgrpid=136191868999&hvadid=669810086624&hvdev=c&hvlocphy=1005381&hvnetw=g&hvqmt=b&hvrand=3370222470617312728&hvtargid=kwd-300340335761&hydadcr=3183_2378895&mcid=426386527c1f3f1fa9ded32d0e66be9e&tag=egtxtgostdde-21&ref=pd_sl_65kme0uvfd_b")

### **Extract the Mobiles' Data**

In [ ]:
# Get Mobiles Phones' data
mobile_names = driver.find_elements("xpath", "//h2[@class=\"a-size-base-plus a-spacing-none a-color-base a-text-normal\"]")
mobile_ratings = driver.find_elements("xpath", "//span/a[@class=\"a-popover-trigger a-declarative\"]")
mobile_items_bought = driver.find_elements("xpath", "//a[@class=\"a-link-normal s-underline-text s-underline-link-text s-link-style\"]/span[@class=\"a-size-base s-underline-text\"]")
mobile_prices = driver.find_elements("xpath", "//div[@class=\"a-section a-spacing-none a-spacing-top-small s-price-instructions-style\"]") # Split at EGP
mobile_image_link = driver.find_elements("xpath", "//div/img[@class=\"s-image\"]") 

### **Some Helper Functions**

In [100]:
def get_image_by_url(url):
    '''
    A funcion that downloads an image locally from the web. 

    Args:
        url(str) : the url of the image on the web.
        
    Return:
        file_name(str) : Name of image after locally being dowloaded
    '''

    # Download the image locally
    file_name = wget.download(url)

    return file_name

In [92]:
def move_image(file_name):
    '''
    A funcion that moves the downloaded image to a specified folder 

    Args:
        file_name(str) : Name of image after locally being dowloaded
        
    Return:
        None
    '''
    # Full path to the destination
    prefix = 'D:\ITI\ITI _Content _AI&ML_Track\Data Exploration & Visualization\Tasks\Images'
    destination = os.path.join(prefix, file_name)

    # Check if the file was downloaded
    if not os.path.exists(file_name):
        print(f"File not found after download: {file_name}")
        return

    # Try moving the file
    try:
        shutil.move(file_name, destination)
        print(f"Moved: {file_name} to {destination}")
        
    except Exception as e:
        print(f"Error moving file: {e}")

In [93]:
def encode_image(image_path):
    '''
    A funcion that moves the downloaded image to a specified folder 

    Args:
        image_path (str) : The final path of the image after locally being dowloaded.
        
    Return:
        encoded_string (str) : The base 64 encoded image, to be saved in the csv file.
    '''
    with open(image_path, "rb") as img_file:
        encoded_string= base64.b64encode(img_file.read())
        #print(encoded_string.decode('utf-8'))
        return encoded_string

### **Map the extracted data to python dictionary**

In [101]:
# Create a dictionary to store Mobile Data
mobile_data = {'Image': [],
               'Image Link': [],
               'Name': [],
               'Properties': [],
               'Rating': [],
               'Quantity Bought': [],
               'Price After Discount': []}

In [102]:
for idx, zipped_mobile_info in enumerate(zip(mobile_names, mobile_ratings, mobile_items_bought, mobile_prices, mobile_image_link)):


    mobile_title, mobile_rating, items_bought, mobile_price, mobile_img_link = zipped_mobile_info     # Get the phone's data


    # Visualize the extracted Data
    print(f'##################################### Mobile {idx+1} Before Refinning Data #####################################')
    print(f'{idx+1}. Title:', mobile_title.text)
    print(f'{idx+1}. Rating:', mobile_rating.accessible_name)
    print(f'{idx+1}. Items Bought:', items_bought.text)
    print(f'{idx+1}. Price:', mobile_price.text)
    print(f'{idx+1}. Image Link:', mobile_img_link.get_attribute("src"))


    # Refine the data to get the exact information required
    items_bought = items_bought.text
    mobile_name = mobile_title.text.split(',')[0]
    mobile_properties = mobile_title.text.split(',')[1:]
    mobile_rating = mobile_rating.accessible_name.split(',')[0]
    mobile_price_after_discount = mobile_price.text
    mobile_source_image_link = mobile_img_link.get_attribute("src")

    # Save the image & then encode it to save in the csv file
    file_name = wget.download(mobile_source_image_link)
    encoded_image = encode_image(file_name)
    move_image(file_name)       # move to images folder


    # Remove that Limited Time Deal Text, and other unneceesary text (get the first item in the list that you will encounter)
    mobile_price_after_discount =  mobile_price_after_discount.split('EGP')
    for item in mobile_price_after_discount[1:]:
        item = item.strip()
        if (item != ''):
            mobile_price_after_discount = 'EGP' + item.split('\n')[0]       
            break


    print(f'##################################### Mobile {idx+1} After Refinning Data #####################################')
    print(f'{idx+1}. Name:', mobile_name)
    print(f'{idx+1}. Properties:', mobile_properties)
    print(f'{idx+1}. Items Bought:', items_bought)
    print(f'{idx+1}. Rating:', mobile_rating)
    print(f'{idx+1}. Price After Discount:', mobile_price_after_discount)
    print(f'{idx+1}. Image Link:', mobile_source_image_link)
    print(f'{idx+1}. Encoded Image:', encoded_image)


    # Save the Data in the dictionary
    mobile_data['Name'].append(mobile_name)
    mobile_data['Properties'].append(mobile_properties)
    mobile_data['Rating'].append(mobile_rating)
    mobile_data['Quantity Bought'].append(items_bought)
    mobile_data['Price After Discount'].append(mobile_price_after_discount)
    mobile_data['Image Link'].append(mobile_source_image_link)
    mobile_data['Image'].append(encoded_image)

##################################### Mobile 1 Before Refinning Data #####################################
1. Title: Nokia c21 plus android smartphone,dual sim,3 gb ram,64 gb memory,6.517",hd+ lcd with v-notch,android 11 (go edition), fingerprint,face unlock,proximity sensor- dark cyan
1. Rating: 3.0 out of 5 stars, rating details
1. Items Bought: 14
1. Price: EGP3,595
00
1. Image Link: https://m.media-amazon.com/images/I/81AuwSoF9yL._AC_UL320_.jpg
Moved: 81AuwSoF9yL._AC_UL320_.jpg to D:\ITI\ITI _Content _AI&ML_Track\Data Exploration & Visualization\Tasks\Images\81AuwSoF9yL._AC_UL320_.jpg
##################################### Mobile 1 After Refinning Data #####################################
1. Name: Nokia c21 plus android smartphone
1. Properties: ['dual sim', '3 gb ram', '64 gb memory', '6.517"', 'hd+ lcd with v-notch', 'android 11 (go edition)', ' fingerprint', 'face unlock', 'proximity sensor- dark cyan']
1. Items Bought: 14
1. Rating: 3.0 out of 5 stars
1. Price After Discount: E

### **Convert the python dictionary to DataFrame**

In [103]:
mobile_data_df = pd.DataFrame(mobile_data)

In [104]:
mobile_data_df.head()

,Image,Image Link,Name,Properties,Rating,Quantity Bought,Price After Discount
0,b'/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUFBQkGCQkJ...,https://m.media-amazon.com/images/I/81AuwSoF9y...,Nokia c21 plus android smartphone,"[dual sim, 3 gb ram, 64 gb memory, 6.517"", hd+...",3.0 out of 5 stars,14,"EGP3,595"
1,b'/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUFBQkGCQkJ...,https://m.media-amazon.com/images/I/413YOEm4KK...,Basic 105 Mobile Phone,"[ Dual SIM, Keypad Design, Black]",5.0 out of 5 stars,1,EGP519
2,b'/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUFBQkGCQkJ...,https://m.media-amazon.com/images/I/21HPpPXvVt...,Basic 106 Mobile Phone,"[ Dual SIM, Keypad Design, 1.8 inch Display,...",4.1 out of 5 stars,51,EGP539
3,b'/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUFBQkGCQkJ...,https://m.media-amazon.com/images/I/31P+3EFo3l...,Xiaomi Redmi Note 14 Smartphone,"[ 6 + 128 GB, Black| 18 Month manufacturer wa...",3.8 out of 5 stars,49,"EGP8,888"
4,b'/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUFBQkGCQkJ...,https://m.media-amazon.com/images/I/51EhoXkyKc...,Redmi 13 Mobile,[ Midnight Black (6GB Ram+128GB) |Mediatek hel...,3.8 out of 5 stars,57,"EGP6,399"


### **Save the DataFrame AS CSV File**

In [105]:
mobile_data_df.to_csv('mobile_data.csv', index=False)